In [54]:
import re
def clean_text(text):
    text = str(text)
    text = text.lower()
    text = text.replace('"', '')
    text = re.sub(r'[^\w\s]', '', text)
    text = text.strip()
    return text

In [55]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pickle
import os

def train_model():
    df = pd.read_csv(r'C:\Users\pc\Documents\SholaMatch\backend\scholamatch_BE\ML\RawData\reel comment b200.csv')
    #Clean data
    df['Comments'] = df['Comments'].apply(clean_text)
    #X & y's
    X = df['Comments']
    y = df['Sentiment']
    #Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    #Build pipeline
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000)),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    #train
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    print(classification_report(y_test, y_pred))
    #save model
    model_path = 'sentimentClassifier_model.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(pipeline, f)
    print("Model Saved!")
    

In [56]:
if __name__ == "__main__":
    train_model()

              precision    recall  f1-score   support

          -1       0.93      0.70      0.80        20
           0       0.76      0.84      0.80        19
           1       0.74      0.85      0.79        20

    accuracy                           0.80        59
   macro avg       0.81      0.80      0.80        59
weighted avg       0.81      0.80      0.80        59

Model Saved!


In [2]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

def visualize_results(y_test, y_pred):
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Sentiment Classifier Results', fontsize=16, fontweight='bold')
    
    # 1 - Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negative', 'Neutral', 'Positive'],
                yticklabels=['Negative', 'Neutral', 'Positive'],
                ax=axes[0])
    axes[0].set_title('Confusion Matrix')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')
    
    # 2 - Precision, Recall, F1 bar chart
    from sklearn.metrics import precision_recall_fscore_support
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred)
    
    x = np.arange(3)
    width = 0.25
    labels = ['Negative', 'Neutral', 'Positive']
    
    axes[1].bar(x - width, precision, width, label='Precision', color='steelblue')
    axes[1].bar(x, recall, width, label='Recall', color='coral')
    axes[1].bar(x + width, f1, width, label='F1-Score', color='mediumseagreen')
    
    axes[1].set_title('Precision, Recall & F1-Score')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels)
    axes[1].set_ylim(0, 1.1)
    axes[1].legend()
    axes[1].set_ylabel('Score')
    
    plt.tight_layout()
    plt.savefig('sentiment_results.png', dpi=150)
    plt.show()
    print("Chart saved!")

    visualize_results(y_test, y_pred)